In [ ]:
import requests
import numpy as np
import json

API_URL = "https://isl-llms.ethz.ch/api"
API_KEY = '{YOUR_API_KEY}'

# Tutorial to interact with the Watermark API using Python

You can visit https://isl-llms.ethz.ch/docs for an interactive documentation. Click "Authorize" to introduce your API-key and use the endpoints directly from the documentation.

Read the assignment for details on this task.

### Retrieve your watermarked token ids

Each student has a different set of tokenized strings that were watermarked with their secret watermarking keys. You can obtain yours by querying the endpoint `/tokens/` as follows.

In [ ]:
def get_tokens():
    url_get_tokens = API_URL + "/tokens"

    payload = {}
    headers = {
      'x-api-key': API_KEY
    }

    return requests.request("GET", url_get_tokens, headers=headers, data=payload).json()

In [ ]:
tokens = get_tokens()

In [ ]:
# Get the token ids for each endpoint. Remember you must use them with the corect endpoint!
pvalue_token_ids = [i for i in tokens if i["endpoint"]=="pvalue"]
bool_token_ids = [i for i in tokens if i["endpoint"]=="bool"]

In [ ]:
bool_token_ids[0]

In [ ]:
# You can obtain a list of integers by using the following function
def tokens_to_array(token_list_str: str) -> np.ndarray:
    return np.array([int(i) for i in token_list_str.split(",")])

tokens_to_array(bool_token_ids[0]["tokens"])

### Check if a specific list of tokens is watermarked with your keys


#### p-value API

In [ ]:
# Use this function to transform a list of integers into a valid string you can submit through the API
def tokens_to_string(token_list: np.array):
    return ",".join([str(i) for i in token_list])

tokens_to_string(tokens_to_array(bool_token_ids[0]["tokens"]))

In [ ]:
assert tokens_to_string(tokens_to_array(bool_token_ids[0]["tokens"])) == (bool_token_ids[0]["tokens"])

In [ ]:
# Use this function to validate your tokens are valid
def validate_tokens(token_list_str: str):
    assert isinstance(token_list_str, str), "Your list must be a string of comma-separated integers"
    assert len(token_list_str)>0, "Your list is empty"

    # Split the string by commas
    parts = token_list_str.split(',')

    # Check if any part is empty or not an integer
    for part in parts:
        assert part.isdigit(), "Some entries in your list are not integers or are empty"

    assert not (len(parts) < 40 or len(parts) > 1000), "Your list must contain at least 40 and at most 1000 tokens. It is {} tokens long".format(len(parts))

In [ ]:
def get_pvalue(tokens: str):
    validate_tokens(tokens)

    url_get_pvalue = API_URL + "/watermark/get_pvalue"

    payload = json.dumps({
      "tokens": tokens
    })

    headers = {
      'Content-Type': 'application/json',
      'x-api-key': API_KEY
    }

    response = requests.request("POST", url_get_pvalue, headers=headers, data=payload)

    return response.json()

In [ ]:
get_pvalue(pvalue_token_ids[0]["tokens"])

#### boolean API

In [ ]:
def get_bool(tokens: str):
    validate_tokens(tokens)

    url_get_bool = API_URL + "/watermark/get_bool"

    payload = json.dumps({
      "tokens": tokens
    })

    headers = {
      'Content-Type': 'application/json',
      'x-api-key': API_KEY
    }

    response = requests.request("POST", url_get_bool, headers=headers, data=payload)

    return response.json()

In [ ]:
get_bool(bool_token_ids[0]["tokens"])

### Check your quota

In [ ]:
url = "https://isl-llms.ethz.ch/check-limits/{}".format(API_KEY)

payload = {}
headers = {}

response = requests.request("GET", url, headers=headers, data=payload)

print(response.text)

# Export your solution

To save your results, you can use the code below, which will save the file in Colab's temporary storage (or locally, if you're not using Colab), or on your Google Drive. If you save it on Colab's temporary storage, you can download it from there (see the file system icon on the left).

In [ ]:
# @title Download lab files

import sys

!rm -rf llm_lab

![ ! -d 'llm_lab' ] && git clone https://github.com/ethz-privsec/llm_lab.git
%cd llm_lab
!git pull https://github.com/ethz-privsec/llm_lab.git
%cd ..
if "llm_lab" not in sys.path:
  sys.path.append("llm_lab")

In [ ]:
from llm_lab.utils import get_solution_path, is_valid_student_id

#@markdown Check this box if you want to save your results on Google Drive. Otherwise they'll be
#@markdown saved on the ephimeral Colab storage. The storage will be deleted with the runtime,
#@markdown so REMEMBER TO DOWNLOAD THE FILES before you close the tab!
SAVE_ON_DRIVE = True # @param {"type":"boolean"}

#@markdown The number on your Legi (Student ID card). It's in the format 'dd-ddd-ddd'
STUDENT_ID = "00-000-000"  # @param {"type":"string","placeholder":"00-000-000"}

assert is_valid_student_id(STUDENT_ID), "Student ID should have the format 'dd-ddd-ddd'"

SOLUTIONS_PATH = get_solution_path(STUDENT_ID, SAVE_ON_DRIVE)

In [ ]:
# Create Q2 key.txt with your API key
with open(SOLUTIONS_PATH / "Q2_key.txt", "w") as f:
  f.write(API_KEY)

In [ ]:
# Set your suffixes here

YOUR_PVALUE_SUFFIX = "1,2,3,4,5,6"
YOUR_BOOL_SUFFIX = "7,8,9,10,11,12"

raise NotImplementedError("Replace the above suffixes with the strings you found")

# Make sure they are numpy arrays
YOUR_PVALUE_SUFFIX = np.array(YOUR_PVALUE_SUFFIX.split(","), dtype=int)
YOUR_BOOL_SUFFIX = np.array(YOUR_BOOL_SUFFIX.split(","), dtype=int)

assert isinstance(YOUR_PVALUE_SUFFIX, np.ndarray) and isinstance(YOUR_PVALUE_SUFFIX[0], np.int64)
assert isinstance(YOUR_BOOL_SUFFIX, np.ndarray) and isinstance(YOUR_BOOL_SUFFIX[0], np.int64)

### Validate your suffixes (optional)
Run this cell to check that your suffixes pass all 7 strings before exporting. This uses **14 API calls** (7 for pvalue + 7 for bool).

In [ ]:
# Validate your suffixes against all 7 strings for each endpoint
tokens = get_tokens()
pvalue_token_ids = [t for t in tokens if t["endpoint"] == "pvalue"]
bool_token_ids = [t for t in tokens if t["endpoint"] == "bool"]

pvalue_suffix_str = tokens_to_string(YOUR_PVALUE_SUFFIX)
bool_suffix_str = tokens_to_string(YOUR_BOOL_SUFFIX)

print("=== Validating pvalue suffix against all 7 strings ===")
for i, t in enumerate(pvalue_token_ids):
    combined = t["tokens"] + "," + pvalue_suffix_str
    result = get_pvalue(combined)
    print(f"  String {i+1}/7: {result}")

print()

print("=== Validating bool suffix against all 7 strings ===")
for i, t in enumerate(bool_token_ids):
    combined = t["tokens"] + "," + bool_suffix_str
    result = get_bool(combined)
    print(f"  String {i+1}/7: {result}")

### Export your suffixes

In [ ]:
with open(SOLUTIONS_PATH / 'Q2_pvalue.npy', 'wb') as f:
    np.save(f, YOUR_PVALUE_SUFFIX)

with open(SOLUTIONS_PATH / 'Q2_bool.npy', 'wb') as f:
    np.save(f, YOUR_BOOL_SUFFIX)

print("Saved Q2_pvalue.npy and Q2_bool.npy")